# Dynamic Electricity Tariffs - Interactive Visualization

This notebook allows you to explore dynamic electricity tariffs for different providers in 2025.
- **Consumer Tariffs**: Electricity costs per kWh (market price + energy tax + margin) × VAT
- **Feed-in Tariffs**: Compensation rates for electricity fed back to the grid

## 1. Import Required Libraries

In [92]:
import pandas as pd
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import numpy as np

# Configure plotly for better display
import plotly.io as pio
pio.renderers.default = "notebook"

## 2. Load Tariff Data

In [93]:
# Define tariff data directory
tariff_dir = Path.cwd().parent / 'Dynamic_contracts' / '2025'

# Get list of providers
provider_dirs = sorted([d.name for d in tariff_dir.iterdir() if d.is_dir()])
print(f"Found {len(provider_dirs)} providers:")
print(", ".join(provider_dirs))

Found 19 providers:
Budget Energie, Coolblue Energie, Energiedirect, Energiek, Frank Energie, Greenchoice, HalloStroom, Innova Energie, Mega Energie, NextEnergy, Oxxio, Powerpeers, Pure Energie, Tibber, Vandebron, Vattenfall, Vrijopnaam, Zonneplan, easyEnergy


In [94]:
# Load ENTSOE market prices from parquet file
entsoe_prices_file = Path.cwd().parent / 'input' / 'entsoe_prices_NL_2025_2026_combined.parquet'
market_prices_df = pd.read_parquet(entsoe_prices_file)
market_prices_df['ts_utc'] = pd.to_datetime(market_prices_df['ts_utc'])
market_prices_df['price_kwh'] = market_prices_df['price'] #EUR/kWh
market_prices_2025 = market_prices_df[market_prices_df['ts_utc'].dt.year == 2025].copy()

print(f"Loaded {len(market_prices_2025)} ENTSOE market price records for 2025")
print(f"Date range: {market_prices_2025['ts_utc'].min()} to {market_prices_2025['ts_utc'].max()}")

Loaded 35029 ENTSOE market price records for 2025
Date range: 2025-01-01 00:15:00+00:00 to 2025-12-31 23:45:00+00:00


In [95]:
def load_provider_tariffs(provider_name):
    """Load consumer and feed-in tariffs for a provider, with ENTSOE market prices."""
    provider_path = tariff_dir / provider_name
    
    # Load consumer tariffs (timeseries_2025.json)
    consumer_file = provider_path / 'timeseries_2025.json'
    with open(consumer_file, 'r') as f:
        consumer_data = json.load(f)
    
    # Load feed-in tariffs (feedin_timeseries_2025.json)
    feedin_file = provider_path / 'feedin_timeseries_2025.json'
    with open(feedin_file, 'r') as f:
        feedin_data = json.load(f)
    
    # Create DataFrames
    consumer_df = pd.DataFrame({
        'timestamp': pd.to_datetime(consumer_data['data']['timestamp']),
        'total_cost_kwh': consumer_data['data']['total_cost_kwh']
    })
    
    feedin_df = pd.DataFrame({
        'timestamp': pd.to_datetime(feedin_data['data']['timestamp']),
        'feedin_rate_kwh': feedin_data['data']['feedin_rate_kwh']
    })
    
    # Merge market prices from parquet file
    market_prices_subset = market_prices_2025[['ts_utc', 'price_kwh']].copy()
    market_prices_subset.rename(columns={'ts_utc': 'timestamp'}, inplace=True)
    
    # Merge all data on timestamp
    combined_df = pd.merge(consumer_df, feedin_df, on='timestamp', how='left')
    combined_df = pd.merge(combined_df, market_prices_subset, on='timestamp', how='left')
    
    return combined_df, consumer_data, feedin_data

# Test loading first provider
print(f"Testing data load for {provider_dirs[0]}...")
test_df, test_consumer, test_feedin = load_provider_tariffs(provider_dirs[0])
print(f"  - Loaded {len(test_df)} records")
print(f"  - Date range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
print(f"  - Market price (ENTSOE) - Min: €{test_df['price_kwh'].min():.6f}, Max: €{test_df['price_kwh'].max():.6f}/kWh")
print(f"  - Consumer total cost - Avg: €{test_df['total_cost_kwh'].mean():.6f}/kWh")
print(f"  - Feed-in rate: €{test_feedin['feedin_rate_kwh']:.6f}/kWh")

Testing data load for Budget Energie...
  - Loaded 35029 records
  - Date range: 2025-01-01 00:15:00+00:00 to 2025-12-31 23:45:00+00:00
  - Market price (ENTSOE) - Min: €-0.350000, Max: €0.523470/kWh
  - Consumer total cost - Avg: €0.237441/kWh
  - Feed-in rate: €0.020990/kWh


## 3. Create Interactive Provider Selector

In [96]:
# Create provider dropdown selector
provider_dropdown = widgets.Dropdown(
    options=provider_dirs,
    value=provider_dirs[0],
    description='Provider:',
    style={'description_width': '100px'}
)

# Create output widget for the plot
output_plot = widgets.Output()

# Create statistics output
output_stats = widgets.Output()

print("Interactive Provider Selector created")

Interactive Provider Selector created


## 4. Plot Tariffs as Time Series

In [97]:
def plot_provider_tariffs(provider_name):
    """Create an interactive time series plot for provider tariffs with ENTSOE market prices."""
    
    # Load data
    df, consumer_info, feedin_info = load_provider_tariffs(provider_name)
    
    # Create subplots with secondary y-axis
    fig = make_subplots(
        rows=1, cols=1,
        specs=[[{"secondary_y": True}]]
    )
    
    # Add ENTSOE market price (left y-axis, prominent)
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['price_kwh'],
            mode='lines',
            name='ENTSOE Market Price',
            line=dict(color='#3498DB', width=2.5),
            hovertemplate='<b>Time:</b> %{x}<br><b>Market Price:</b> €%{y:.4f}/kWh<extra></extra>'
        ),
        secondary_y=False
    )
    
    # Add consumer tariff line (left y-axis)
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['total_cost_kwh'],
            mode='lines',
            name='Consumer Tariff (Total Cost)',
            line=dict(color='#E74C3C', width=2),
            hovertemplate='<b>Time:</b> %{x}<br><b>Consumer Total:</b> €%{y:.4f}/kWh<extra></extra>'
        ),
        secondary_y=False
    )
    
    # Add feed-in tariff line (right y-axis)
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['feedin_rate_kwh'],
            mode='lines',
            name='Feed-in Tariff',
            line=dict(color='#27AE60', width=2),
            hovertemplate='<b>Time:</b> %{x}<br><b>Feed-in:</b> €%{y:.4f}/kWh<extra></extra>'
        ),
        secondary_y=True
    )
    
    # Update layout with titles and labels
    fig.update_layout(
        title=f'<b>{provider_name} - Dynamic Tariffs 2025 (with ENTSOE Market Price)</b>',
        hovermode='x unified',
        template='plotly_white',
        height=600,
        font=dict(size=11),
        margin=dict(t=100, b=80, l=80, r=80)
    )
    
    # Update x-axis
    fig.update_xaxes(title_text='Date', tickformat='%Y-%m-%d')
    
    # Update y-axes
    fig.update_yaxes(title_text='<b>Market Price & Consumer Tariff (EUR/kWh)</b>', secondary_y=False)
    fig.update_yaxes(title_text='<b>Feed-in Tariff (EUR/kWh)</b>', secondary_y=True)
    
    return fig, df, consumer_info, feedin_info

# Test the plot function
print("Testing plot function with ENTSOE market prices...")

Testing plot function with ENTSOE market prices...


## 5. Display Selected Provider Data

Use the dropdown below to select a provider and view their tariffs as an interactive time series plot.

In [98]:
def update_display(change):
    """Update the plot and statistics when provider is changed."""
    provider = change['new']
    
    # Generate plot and data
    fig, df, consumer_info, feedin_info = plot_provider_tariffs(provider)
    
    # Calculate statistics
    consumer_tariff = consumer_info['data']['total_cost_kwh']
    feedin_rate = feedin_info['feedin_rate_kwh']
    market_prices = df['price_kwh'].dropna()
    
    # Update plot output
    with output_plot:
        clear_output(wait=True)
        fig.show()
    
    # Update statistics output
    with output_stats:
        clear_output(wait=True)
        print(f"\n📊 Statistics for {provider}:\n")
        
        print(f"ENTSOE Market Price (from parquet file):")
        print(f"  - Minimum: €{market_prices.min():.6f}/kWh")
        print(f"  - Maximum: €{market_prices.max():.6f}/kWh")
        print(f"  - Average: €{market_prices.mean():.6f}/kWh")
        
        print(f"\nConsumer Tariff (Total Cost):")
        print(f"  - Minimum: €{min(consumer_tariff):.6f}/kWh")
        print(f"  - Maximum: €{max(consumer_tariff):.6f}/kWh")
        print(f"  - Average: €{np.mean(consumer_tariff):.6f}/kWh")
        print(f"  - Margin: €{consumer_info['margin']:.6f}/kWh")
        print(f"  - Markup over market: {((np.mean(consumer_tariff) / market_prices.mean()) - 1) * 100:.1f}%")
        
        print(f"\nFeed-in Tariff:")
        print(f"  - Rate: €{feedin_rate:.6f}/kWh")
        if feedin_rate == 0:
            print(f"  - ⚠️  No feed-in compensation offered")

# Attach the callback to the dropdown
provider_dropdown.observe(update_display, names='value')

# Display the interactive interface
print("=" * 70)
print("DYNAMIC ELECTRICITY TARIFFS - INTERACTIVE VISUALIZER")
print("=" * 70)
print()
display(provider_dropdown)
display(output_stats)
display(output_plot)

# Trigger initial plot
update_display({'new': provider_dropdown.value})

DYNAMIC ELECTRICITY TARIFFS - INTERACTIVE VISUALIZER



Dropdown(description='Provider:', options=('Budget Energie', 'Coolblue Energie', 'Energiedirect', 'Energiek', …

Output()

Output()

## Additional Features

### Export Data to CSV

Export selected provider's tariff data to CSV for further analysis:

In [99]:
# Create export button
export_button = widgets.Button(
    description='Export to CSV',
    tooltip='Export current provider data to CSV',
    button_style='info'
)

output_export = widgets.Output()

def on_export_clicked(b):
    """Export current provider data to CSV."""
    provider = provider_dropdown.value
    df, _, _ = load_provider_tariffs(provider)
    
    # Create output filename
    filename = f"{provider.replace(' ', '_')}_tariffs_2025.csv"
    filepath = tariff_dir.parent / filename
    
    # Export to CSV
    df.to_csv(filepath, index=False)
    
    with output_export:
        clear_output(wait=True)
        print(f"✅ Data exported successfully!")
        print(f"   File: {filename}")
        print(f"   Location: {filepath}")
        print(f"   Records: {len(df)}")

export_button.on_click(on_export_clicked)

print("Export feature ready")
print("\nClick the button below to export the selected provider's tariff data:")
display(export_button)
display(output_export)

Export feature ready

Click the button below to export the selected provider's tariff data:


Button(button_style='info', description='Export to CSV', style=ButtonStyle(), tooltip='Export current provider…

Output()

## Summary

This interactive visualization allows you to:

1. **Select any provider** from the dropdown menu to view their tariffs
2. **View three tariff components** in the time series plot:
   - **Consumer Tariff (red line)**: Total electricity cost including market price, energy tax, margin, and VAT
   - **Market Price (orange dashed)**: The hourly ENTSOE electricity market price (left y-axis)
   - **Feed-in Tariff (green line)**: Compensation rate for electricity fed back to the grid (right y-axis)
3. **Explore detailed statistics** including minimum, maximum, and average rates for each provider
4. **Export data to CSV** for further analysis

### Key Metrics:
- **19 dynamic electricity providers** included
- **35,029 data points** per provider (15-minute intervals throughout 2025)
- **Hourly market prices** from ENTSOE combined with provider margins
- **Feed-in compensation rates** for solar/renewable installations

## Summary

This interactive visualization allows you to:

1. **Select any provider** from the dropdown menu to view their tariffs
2. **View three tariff components** in the time series plot:
   - **Consumer Tariff (red line)**: Total electricity cost including market price, energy tax, margin, and VAT
   - **Market Price (orange dashed)**: The hourly ENTSOE electricity market price (left y-axis)
   - **Feed-in Tariff (green line)**: Compensation rate for electricity fed back to the grid (right y-axis)
3. **Explore detailed statistics** including minimum, maximum, and average rates for each provider
4. **Export data to CSV** for further analysis

### Key Metrics:
- **19 dynamic electricity providers** included
- **35,029 data points** per provider (15-minute intervals throughout 2025)
- **Hourly market prices** from ENTSOE combined with provider margins
- **Feed-in compensation rates** for solar/renewable installations

In [100]:
# Create export button
export_button = widgets.Button(
    description='Export to CSV',
    tooltip='Export current provider data to CSV',
    button_style='info'
)

output_export = widgets.Output()

def on_export_clicked(b):
    """Export current provider data to CSV."""
    provider = provider_dropdown.value
    df, _, _ = load_provider_tariffs(provider)
    
    # Create output filename
    filename = f"{provider.replace(' ', '_')}_tariffs_2025.csv"
    filepath = tariff_dir.parent / filename
    
    # Export to CSV
    df.to_csv(filepath, index=False)
    
    with output_export:
        clear_output(wait=True)
        print(f"✅ Data exported successfully!")
        print(f"   File: {filename}")
        print(f"   Location: {filepath}")
        print(f"   Records: {len(df)}")

export_button.on_click(on_export_clicked)

print("Export feature ready")
print("\nClick the button below to export the selected provider's tariff data:")
display(export_button)
display(output_export)

Export feature ready

Click the button below to export the selected provider's tariff data:


Button(button_style='info', description='Export to CSV', style=ButtonStyle(), tooltip='Export current provider…

Output()

## Additional Features

### Export Data to CSV

Export selected provider's tariff data to CSV for further analysis:

In [101]:
def update_display(change):
    """Update the plot and statistics when provider is changed."""
    provider = change['new']
    
    # Generate plot and data
    fig, df, consumer_info, feedin_info = plot_provider_tariffs(provider)
    
    # Calculate statistics
    consumer_tariff = consumer_info['data']['total_cost_kwh']
    feedin_rate = feedin_info['feedin_rate_kwh']
    
    # Update plot output
    with output_plot:
        clear_output(wait=True)
        fig.show()
    
    # Update statistics output
    with output_stats:
        clear_output(wait=True)
        print(f"\n📊 Statistics for {provider}:\n")
        print(f"Consumer Tariff (Total Cost):")
        print(f"  - Minimum: €{min(consumer_tariff):.6f}/kWh")
        print(f"  - Maximum: €{max(consumer_tariff):.6f}/kWh")
        print(f"  - Average: €{np.mean(consumer_tariff):.6f}/kWh")
        print(f"  - Margin: €{consumer_info['margin']:.6f}/kWh")
        
        print(f"\nFeed-in Tariff:")
        print(f"  - Rate: €{feedin_rate:.6f}/kWh")
        if feedin_rate == 0:
            print(f"  - ⚠️  No feed-in compensation offered")

# Attach the callback to the dropdown
provider_dropdown.observe(update_display, names='value')

# Display the interactive interface
print("=" * 70)
print("DYNAMIC ELECTRICITY TARIFFS - INTERACTIVE VISUALIZER")
print("=" * 70)
print()
display(provider_dropdown)
display(output_stats)
display(output_plot)

# Trigger initial plot
update_display({'new': provider_dropdown.value})

DYNAMIC ELECTRICITY TARIFFS - INTERACTIVE VISUALIZER



Dropdown(description='Provider:', options=('Budget Energie', 'Coolblue Energie', 'Energiedirect', 'Energiek', …

Output(outputs=({'name': 'stdout', 'text': '\n📊 Statistics for Budget Energie:\n\nENTSOE Market Price (from pa…

Output(outputs=({'output_type': 'display_data', 'data': {'text/html': '        <script type="text/javascript">…

## 5. Display Selected Provider Data

Use the dropdown below to select a provider and view their tariffs as an interactive time series plot.

In [102]:
def plot_provider_tariffs(provider_name):
    """Create an interactive time series plot for provider tariffs."""
    
    # Load data
    df, consumer_info, feedin_info = load_provider_tariffs(provider_name)
    
    # Create subplots with secondary y-axis
    fig = make_subplots(
        rows=1, cols=1,
        specs=[[{"secondary_y": True}]]
    )
    
    # Add consumer tariff line (left y-axis)
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['total_cost_kwh'],
            mode='lines',
            name='Consumer Tariff (Total Cost)',
            line=dict(color='#E74C3C', width=2),
            hovertemplate='<b>Time:</b> %{x}<br><b>Cost:</b> €%{y:.4f}/kWh<extra></extra>'
        ),
        secondary_y=False
    )
    
    # Add market price component (lighter version on left y-axis)
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['market_price_kwh'],
            mode='lines',
            name='Market Price',
            line=dict(color='#F39C12', width=1, dash='dash'),
            opacity=0.6,
            hovertemplate='<b>Time:</b> %{x}<br><b>Market Price:</b> €%{y:.4f}/kWh<extra></extra>'
        ),
        secondary_y=False
    )
    
    # Add feed-in tariff line (right y-axis)
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['feedin_rate_kwh'],
            mode='lines',
            name='Feed-in Tariff',
            line=dict(color='#27AE60', width=2),
            hovertemplate='<b>Time:</b> %{x}<br><b>Feed-in:</b> €%{y:.4f}/kWh<extra></extra>'
        ),
        secondary_y=True
    )
    
    # Update layout with titles and labels
    fig.update_layout(
        title=f'<b>{provider_name} - Dynamic Tariffs 2025</b>',
        hovermode='x unified',
        template='plotly_white',
        height=600,
        font=dict(size=11),
        margin=dict(t=100, b=80, l=80, r=80)
    )
    
    # Update x-axis
    fig.update_xaxes(title_text='Date', tickformat='%Y-%m-%d')
    
    # Update y-axes
    fig.update_yaxes(title_text='<b>Consumer Tariff (EUR/kWh)</b>', secondary_y=False)
    fig.update_yaxes(title_text='<b>Feed-in Tariff (EUR/kWh)</b>', secondary_y=True)
    
    return fig, df, consumer_info, feedin_info

# Test the plot function
print("Testing plot function...")

Testing plot function...


## 4. Plot Tariffs as Time Series

In [103]:
# Create provider dropdown selector
provider_dropdown = widgets.Dropdown(
    options=provider_dirs,
    value=provider_dirs[0],
    description='Provider:',
    style={'description_width': '100px'}
)

# Create output widget for the plot
output_plot = widgets.Output()

# Create statistics output
output_stats = widgets.Output()

print("Interactive Provider Selector created")

Interactive Provider Selector created


## 3. Create Interactive Provider Selector

In [104]:
def load_provider_tariffs(provider_name):
    """Load consumer and feed-in tariffs for a provider."""
    provider_path = tariff_dir / provider_name
    
    # Load consumer tariffs (timeseries_2025.json)
    consumer_file = provider_path / 'timeseries_2025.json'
    with open(consumer_file, 'r') as f:
        consumer_data = json.load(f)
    
    # Load feed-in tariffs (feedin_timeseries_2025.json)
    feedin_file = provider_path / 'feedin_timeseries_2025.json'
    with open(feedin_file, 'r') as f:
        feedin_data = json.load(f)
    
    # Create DataFrames
    consumer_df = pd.DataFrame({
        'timestamp': pd.to_datetime(consumer_data['data']['timestamp']),
        'market_price_kwh': consumer_data['data']['market_price_kwh'],
        'total_cost_kwh': consumer_data['data']['total_cost_kwh']
    })
    
    feedin_df = pd.DataFrame({
        'timestamp': pd.to_datetime(feedin_data['data']['timestamp']),
        'feedin_rate_kwh': feedin_data['data']['feedin_rate_kwh']
    })
    
    # Merge on timestamp
    combined_df = pd.merge(consumer_df, feedin_df, on='timestamp', how='left')
    
    return combined_df, consumer_data, feedin_data

# Test loading first provider
print(f"Testing data load for {provider_dirs[0]}...")
test_df, test_consumer, test_feedin = load_provider_tariffs(provider_dirs[0])
print(f"  - Loaded {len(test_df)} records")
print(f"  - Date range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
print(f"  - Consumer rate avg: {test_consumer['data']['total_cost_kwh'][0]:.4f} EUR/kWh on avg")
print(f"  - Feed-in rate: {test_feedin['feedin_rate_kwh']:.4f} EUR/kWh")

Testing data load for Budget Energie...
  - Loaded 35029 records
  - Date range: 2025-01-01 00:15:00+00:00 to 2025-12-31 23:45:00+00:00
  - Consumer rate avg: 0.1484 EUR/kWh on avg
  - Feed-in rate: 0.0210 EUR/kWh


In [105]:
# Define tariff data directory
tariff_dir = Path.cwd().parent / 'Dynamic_contracts' / '2025'

# Get list of providers
provider_dirs = sorted([d.name for d in tariff_dir.iterdir() if d.is_dir()])
print(f"Found {len(provider_dirs)} providers:")
print(", ".join(provider_dirs))

Found 19 providers:
Budget Energie, Coolblue Energie, Energiedirect, Energiek, Frank Energie, Greenchoice, HalloStroom, Innova Energie, Mega Energie, NextEnergy, Oxxio, Powerpeers, Pure Energie, Tibber, Vandebron, Vattenfall, Vrijopnaam, Zonneplan, easyEnergy


## 2. Load Tariff Data

In [106]:
import pandas as pd
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import numpy as np

# Configure plotly for better display
import plotly.io as pio
pio.renderers.default = "notebook"

## 1. Import Required Libraries

# Dynamic Electricity Tariffs - Interactive Visualization

This notebook allows you to explore dynamic electricity tariffs for different providers in 2025.
- **Consumer Tariffs**: Electricity costs per kWh (market price + energy tax + margin) × VAT
- **Feed-in Tariffs**: Compensation rates for electricity fed back to the grid